<a href="https://colab.research.google.com/github/fhriads/dissz-cloud-drive/blob/main/Komputasi_Numerik.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================
# Penyelesaian SPL dengan Metode Eliminasi Gauss
# (dengan partial pivoting) - siap dijalankan di Google Colab
# =============================================================
import numpy as np

EPSILON = 1e-12


def gauss_eliminasi(A, b, tampil_langkah=True):
    """Menyelesaikan A.x = b dengan eliminasi Gauss.
    Mengembalikan vektor x, atau None jika tidak ada solusi tunggal."""
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    # Bentuk matriks augmented [A | b]
    M = np.hstack((A, b.reshape(-1, 1)))
    np.set_printoptions(precision=4, suppress=True, linewidth=120)

    if tampil_langkah:
        print("Matriks augmented awal:")
        print(M, "\n")

    # ---------- Tahap 1: Eliminasi maju ----------
    for k in range(n - 1):
        # Partial pivoting: cari baris dengan |nilai| terbesar di kolom k
        p = k + np.argmax(np.abs(M[k:, k]))

        if abs(M[p, k]) < EPSILON:
            print("SPL tidak memiliki solusi tunggal (pivot = 0).")
            return None

        if p != k:
            M[[k, p]] = M[[p, k]]  # tukar baris
            if tampil_langkah:
                print(f"Tukar baris R{k+1} <-> R{p+1}")

        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]
            M[i, k:] = M[i, k:] - m * M[k, k:]
            if tampil_langkah:
                print(f"R{i+1} <- R{i+1} - ({m:.4f}) * R{k+1}")

        if tampil_langkah:
            print(f"\nSetelah eliminasi kolom {k+1}:")
            print(M, "\n")

    if abs(M[n - 1, n - 1]) < EPSILON:
        print("SPL tidak memiliki solusi tunggal (pivot = 0).")
        return None

    # ---------- Tahap 2: Substitusi mundur ----------
    x = np.zeros(n)
    x[n - 1] = M[n - 1, n] / M[n - 1, n - 1]
    for i in range(n - 2, -1, -1):
        jumlah = np.dot(M[i, i + 1:n], x[i + 1:n])
        x[i] = (M[i, n] - jumlah) / M[i, i]

    return x


def baca_input():
    """Membaca SPL dari input pengguna."""
    n = int(input("Masukkan jumlah persamaan/variabel (n): "))
    print(f"\nMasukkan koefisien matriks A ({n} baris, pisahkan dengan spasi):")
    A = []
    for i in range(n):
        baris = list(map(float, input(f"  Baris {i+1}: ").split()))
        if len(baris) != n:
            raise ValueError(f"Baris harus berisi {n} angka.")
        A.append(baris)
    print(f"\nMasukkan vektor konstanta b ({n} angka, pisahkan dengan spasi):")
    b = list(map(float, input("  b: ").split()))
    if len(b) != n:
        raise ValueError(f"Vektor b harus berisi {n} angka.")
    return A, b


def main():
    print("=== SPL - Metode Eliminasi Gauss ===")
    print("1. Gunakan contoh soal")
    print("2. Input sendiri")
    pilihan = input("Pilih (1/2): ").strip()

    if pilihan == "2":
        A, b = baca_input()
    else:
        # Contoh:  2x + y - z  = 8
        #         -3x - y + 2z = -11
        #         -2x + y + 2z = -3      -> solusi: x=2, y=3, z=-1
        A = [[2, 1, -1],
             [-3, -1, 2],
             [-2, 1, 2]]
        b = [8, -11, -3]
        print("\nContoh soal yang digunakan:")
        for i in range(len(b)):
            print("  ", A[i], "| ", b[i])
        print()

    x = gauss_eliminasi(A, b)

    if x is not None:
        print("=== HASIL ===")
        for i, nilai in enumerate(x, start=1):
            print(f"x{i} = {nilai:.6f}")

        # Verifikasi dengan NumPy
        print("\nVerifikasi (numpy.linalg.solve):", np.linalg.solve(A, b))
        print("Residual ||A.x - b|| =", np.linalg.norm(np.array(A) @ x - np.array(b)))


main()

=== SPL - Metode Eliminasi Gauss ===
1. Gunakan contoh soal
2. Input sendiri
Pilih (1/2): 1

Contoh soal yang digunakan:
   [2, 1, -1] |  8
   [-3, -1, 2] |  -11
   [-2, 1, 2] |  -3

Matriks augmented awal:
[[  2.   1.  -1.   8.]
 [ -3.  -1.   2. -11.]
 [ -2.   1.   2.  -3.]] 

Tukar baris R1 <-> R2
R2 <- R2 - (-0.6667) * R1
R3 <- R3 - (0.6667) * R1

Setelah eliminasi kolom 1:
[[ -3.      -1.       2.     -11.    ]
 [  0.       0.3333   0.3333   0.6667]
 [  0.       1.6667   0.6667   4.3333]] 

Tukar baris R2 <-> R3
R3 <- R3 - (0.2000) * R2

Setelah eliminasi kolom 2:
[[ -3.      -1.       2.     -11.    ]
 [  0.       1.6667   0.6667   4.3333]
 [  0.       0.       0.2     -0.2   ]] 

=== HASIL ===
x1 = 2.000000
x2 = 3.000000
x3 = -1.000000

Verifikasi (numpy.linalg.solve): [ 2.  3. -1.]
Residual ||A.x - b|| = 8.881784197001252e-16
